<a href="https://colab.research.google.com/github/Shanmukh-dev/Custom-GPT/blob/main/RAG_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import tiktoken
import torch
from torch import nn
import datasets


tokenizer = tiktoken.get_encoding("gpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [2]:
engine_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py")

with open("model.py", "w") as f:
    f.write(engine_res.text)
    # print(engine_res.text)
print("Downloaded model.py")

# Downloading train_utils.py
train_utils_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py")

with open("train_utils.py", "w") as f:
    f.write(train_utils_res.text)
    # print(engine_res.text)
print("Downloaded train_utils.py")

data_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/data.py")

with open("data.py", "w") as f:
    f.write(data_res.text)
    # print(engine_res.text)
print("Downloaded data.py")
wt_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_30m_it_v1.pth")
with open("tara_n1_30m_it_v1.pth", "wb") as f:
    f.write(wt_res.content)
print("Downloaded modle weights")

Downloaded model.py
Downloaded train_utils.py
Downloaded data.py
Downloaded modle weights


In [3]:
from model import *
from train_utils import *
from data import *


config = GPTConfig(
    vocab_size=tokenizer.n_vocab,
    block_size=1024,
    batch_size=4,
    d_model=256,
    hidden_layers=1024,
    n_heads=4,
    n_layers=6,
)

In [4]:
# import pandas as pd

# data = pd.read_csv("timekeepers_sft_200k.csv")
# df = pd.DataFrame(data)
# # df.head(5)
# eval(df["conversation_messages_json"][0])


import datasets

grounded_rag = datasets.load_dataset("shanaka95/GroundedRAG")
print(len(grounded_rag))

README.md:   0%|          | 0.00/19.6k [00:00<?, ?B/s]

dataset-final.jsonl: reconstructing file:   0%|          |  0.00B / 1.87GB            

dataset-final.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/572598 [00:00<?, ? examples/s]

1


In [5]:
def process_data_in_chunks(dataset, tokenizer, chunk_size=20000, max_samples=400000):
    all_tokens = []
    current_chunk_text = ""
    count = 0

    for sample in dataset:
        query = f"<user>Context:{sample['context']}\nQuestion:{sample['question']}</user>"
        # context = f"<context>{sample['context']}</context>"
        response = f"<assistant>{sample['answer']}</assistant>"
        current_chunk_text += f"{query}\n{response}\n"
        count += 1

        if count % chunk_size == 0:
            # Tokenize the accumulated chunk of text and clear memory
            chunk_tokens = tokenizer.encode(current_chunk_text, allowed_special="all")
            all_tokens.extend(chunk_tokens)
            current_chunk_text = ""
            print(f"Processed {count} samples...")

        if count >= max_samples:
            break

    # Process any remaining text
    if current_chunk_text:
        chunk_tokens = tokenizer.encode(current_chunk_text, allowed_special="all")
        all_tokens.extend(chunk_tokens)

    # Append EOF token
    eof_tokens = tokenizer.encode("<|endoftext|>", allowed_special="all")
    all_tokens.extend(eof_tokens)

    return all_tokens

In [6]:
from tqdm.auto import tqdm
import numpy as np

def create_store_dataset():
    print("Tokenizing dataset in chunks...")
    tokens_list = process_data_in_chunks(grounded_rag["train"], tokenizer, chunk_size=25000, max_samples=400000)

    print("Converting tokens to NumPy array...")
    tokens = np.asarray(tokens_list, dtype=np.uint16)

    print(f"Saving {len(tokens)} tokens to sft.npy...")
    mmap = np.lib.format.open_memmap("sft.npy", dtype=tokens.dtype, mode="w+", shape=tokens.shape)
    mmap[:] = tokens
    mmap.flush()
    print("Successfully saved tokens.")

create_store_dataset()

Tokenizing dataset in chunks...
Processed 25000 samples...
Processed 50000 samples...
Processed 75000 samples...
Processed 100000 samples...
Processed 125000 samples...
Processed 150000 samples...
Processed 175000 samples...
Processed 200000 samples...
Processed 225000 samples...
Processed 250000 samples...
Processed 275000 samples...
Processed 300000 samples...
Processed 325000 samples...
Processed 350000 samples...
Processed 375000 samples...
Processed 400000 samples...
Converting tokens to NumPy array...
Saving 281498441 tokens to sft.npy...
Successfully saved tokens.


In [7]:
from torch.utils.data import Dataset, DataLoader
class MidTrainDataset(Dataset):
    def __init__(self, path, block_size):
        self.data = np.load(path, mmap_mode="r", allow_pickle=True)
        self.block_size = block_size
        self.n_samples = (len(self.data)-1) // self.block_size

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        start = idx * self.block_size
        end = start + self.block_size
        x = self.data[start:end]
        y = self.data[start+1:end+1]

        return (
            torch.from_numpy(x.astype(np.int64)),
            torch.from_numpy(y.astype(np.int64))
        )


In [8]:
full_dataset = MidTrainDataset("sft.npy", config.block_size)

train_split = int(0.9*len(full_dataset))

train_dataset = torch.utils.data.Subset(full_dataset, range(0, train_split))
test_dataset = torch.utils.data.Subset(full_dataset, range(train_split, len(full_dataset)))

train_dataloader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=True, pin_memory=True)

print(len(full_dataset))
print(len(train_dataset))
print(len(test_dataset))

274900
247410
27490


In [9]:
model = CustomGPT(config)
state_dict = torch.load("tara_n1_30m_it_v1.pth", map_location=device)
state_dict = {k.removeprefix("module."):v for k, v in state_dict.items()}
model.load_state_dict(state_dict)

model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler()
calc_params(model)

Total Parameters: 30,783,057
Trainable Parameters: 30,783,057


In [10]:
from tqdm.auto import tqdm

steps = 68800
test_count = 5000
test_steps = 100

train_iter = iter(train_dataloader)

avg_train_loss = 0

for step in tqdm(range(1, steps+1)):
    model.train()
    try:
        if not train_iter:
            train_iter = iter(train_dataloader)

        train_loss = train_step(model, train_iter, loss_fn, optimizer, scaler, device, accumulation_steps=1)
    except StopIteration:
        train_iter = iter(train_dataloader)

        train_loss = train_step(model, train_iter, loss_fn, optimizer, scaler, device, accumulation_steps=1)
    avg_train_loss += train_loss
    if step%test_count == 0:
        avg_train_loss /= test_count

        test_loss = test_step(model, test_dataloader, test_steps, loss_fn, device)

        print(f"Step: {step} | Train loss: {avg_train_loss} | Test loss: {test_loss}")
        avg_train_loss = 0

  0%|          | 0/68800 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Step: 5000 | Train loss: 3.780743940734863 | Test loss: 3.6557325387001036


  0%|          | 0/100 [00:00<?, ?it/s]

Step: 10000 | Train loss: 3.601680478668213 | Test loss: 3.5709503149986266


...

Step: 55000 | Train loss: 3.3602841166973114 | Test loss: 3.3577246713638305


  0%|          | 0/100 [00:00<?, ?it/s]

Step: 60000 | Train loss: 3.359321268129349 | Test loss: 3.396224386692047


  0%|          | 0/100 [00:00<?, ?it/s]

Step: 65000 | Train loss: 3.3280519586086275 | Test loss: 3.295418829917908


In [11]:
prompt = "How much is ATLAS?"
context = torch.tensor(tokenizer.encode(f"<user>{prompt}</user>\n<context>[Document1]This is a lusury watch company that makes custom watches as well. our falgship models are ATLAS - $3000; SOLSTICE - $4000; EXPEDITION - $1500</context>\n<assistant>"), dtype=torch.long, device=device).unsqueeze(0)

with torch.inference_mode():
    generated = model.generate(context, max_new_tokens=512)

result = tokenizer.decode(generated[0].tolist())

print(result)

<user>How much is ATLAS?</user>
<context>[Document1]This is a lusury watch company that makes custom watches as well. our falgship models are ATLAS - $3000; SOLSTICE - $4000; EXPEDITION - $1500</context>
<assistant>Custom watch company delivers fresh 1,000 passingaround 2,000 value and features CHIENCES - you can write an actual price myself from one bank and see how little it turned into a profit!</assistant>
<user>Context:[Document 1] Double Zero ends, the Thank-you that you have brought a series home featuring company team who has cut earnings and deductions, and say... I've said "it's basically a fanatics concept." Just because it really happens: Red Delicious. Just will point you to take hardwood. The end of this story: Red Delicious, this paper was put in the middle of the story to avenge duress, so that critics admired it, a knowledge I own about what the island heck of the dystopian world is currently struggling to fail, and has to apologise for the highest price; in this twist

In [12]:
# def run_chat(model, tokenizer, device, max_new_tokens=256):
#     # Simple array for chat memory
#     chat_history = []

#     print("Chat interface initialized. Type 'exit' or 'quit' to end the session.\n")

#     while True:
#         user_input = input("User: ")
#         if user_input.strip().lower() in ['exit', 'quit']:
#             print("Ending chat session.")
#             break

#         # Add user message to history
#         chat_history.append({"role": "user", "content": user_input})

#         # 1. Format the ENTIRE conversational history
#         formatted_prompt = ""
#         for msg in chat_history:
#             if msg["role"] == "user":
#                 formatted_prompt += f"<user>{msg['content']}</user>\n"
#             else:
#                 formatted_prompt += f"<assistant>{msg['content']}</assistant>\n"

#         # Append the assistant tag to prompt the model to generate the next response
#         formatted_prompt += "<assistant>"

#         # 2. Tokenize the ENTIRE conversational context
#         context = torch.tensor(tokenizer.encode(formatted_prompt, allowed_special="all"), dtype=torch.long, device=device).unsqueeze(0)

#         model.eval()
#         with torch.inference_mode():
#             # Pass the entire conversational token history to the model
#             generated = model.generate(context, max_new_tokens=max_new_tokens)

#         # Decode results
#         full_generation = tokenizer.decode(generated[0].tolist())

#         # Extract only the newly generated response for this turn
#         new_response = full_generation[len(formatted_prompt):]
#         if "</assistant>" in new_response:
#             new_response = new_response.split("</assistant>")[0].strip()
#         else:
#             new_response = new_response.strip()

#         print(f"Assistant: {new_response}\n")

#         # Add assistant response to history so it persists in the next turn's tokenization
#         chat_history.append({"role": "assistant", "content": new_response})

# # Run the updated chat interface
# run_chat(model, tokenizer, device, 512)

In [13]:
torch.save(model.state_dict(), "tara_n1_rag_v1.pth")